In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from src.content_based import ContentBasedRecommender
from src.evaluation import evaluate_recommender

In [2]:
input_dir = Path("../data")

train = pd.read_csv(input_dir / "processed/train.csv")
test = pd.read_csv(input_dir / "processed/test.csv")
movies = pd.read_csv(input_dir / "ml-latest-small/movies.csv")

In [3]:
movies["genres"] = (
    movies["genres"]
    .str.replace("|", " ", regex=False)
    .str.replace("-", "_", regex=False)
    .str.replace("(no genres listed)", "", regex=False)
    .str.strip()
)

In [4]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy
1,2,Jumanji (1995),Adventure Children Fantasy
2,3,Grumpier Old Men (1995),Comedy Romance
3,4,Waiting to Exhale (1995),Comedy Drama Romance
4,5,Father of the Bride Part II (1995),Comedy


In [5]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(movies["genres"])

In [6]:
vectorizer.get_feature_names_out()

array(['action', 'adventure', 'animation', 'children', 'comedy', 'crime',
       'documentary', 'drama', 'fantasy', 'film_noir', 'horror', 'imax',
       'musical', 'mystery', 'romance', 'sci_fi', 'thriller', 'war',
       'western'], dtype=object)

In [7]:
item_content_sim = cosine_similarity(X)


item_sim = pd.DataFrame(item_content_sim, index=movies["movieId"], columns=movies["movieId"])


target_movie = movies["movieId"].iloc[1]

similar_items = item_sim[target_movie].drop(target_movie).sort_values(ascending=False).head(10)


titles = similar_items.index.tolist()
titles

[56915, 2093, 2161, 126142, 104074, 2043, 2162, 60, 1009, 80748]

In [8]:
userId = 1

X_df = pd.DataFrame(X.toarray(), index=movies["movieId"])
user_data = train[train["userId"] == userId]


user_item_vectors = X_df.loc[user_data["movieId"]].values
user_ratings = user_data["rating"].values


user_profile = (user_ratings @ user_item_vectors) / user_ratings.sum()

In [9]:
user_profile

array([0.19921696, 0.20347474, 0.07251729, 0.10268103, 0.17331573,
       0.12446306, 0.        , 0.12537226, 0.11935322, 0.0035411 ,
       0.04150647, 0.        , 0.06498   , 0.0495381 , 0.06231778,
       0.10399376, 0.1169662 , 0.07834659, 0.02449954])

In [10]:
genre_names = vectorizer.get_feature_names_out()
profile_series = pd.Series(user_profile, index=genre_names).sort_values(ascending=False)
print(profile_series)

adventure      0.203475
action         0.199217
comedy         0.173316
drama          0.125372
crime          0.124463
fantasy        0.119353
thriller       0.116966
sci_fi         0.103994
children       0.102681
war            0.078347
animation      0.072517
musical        0.064980
romance        0.062318
mystery        0.049538
horror         0.041506
western        0.024500
film_noir      0.003541
documentary    0.000000
imax           0.000000
dtype: float64


In [11]:
user_profile_2d = user_profile.reshape(1, -1)


user_movie_sim = cosine_similarity(user_profile_2d, X).flatten()


rec_scores = pd.Series(user_movie_sim, index=movies["movieId"])

watched_movies = train[train["userId"] == userId]["movieId"]
rec_scores = rec_scores.drop(index=watched_movies, errors="ignore")


top_10_movie_ids = rec_scores.sort_values(ascending=False).head(10).index


recommended_titles = movies.set_index("movieId").loc[top_10_movie_ids]["title"].tolist()
recommended_titles

['Dragonheart 2: A New Beginning (2000)',
 'Hunting Party, The (2007)',
 'The Great Train Robbery (1978)',
 'Flashback (1990)',
 'Maximum Ride (2016)',
 "Charlie's Angels: Full Throttle (2003)",
 'Diamond Arm, The (Brilliantovaya ruka) (1968)',
 'After the Sunset (2004)',
 'Machete (2010)',
 'Stunt Man, The (1980)']

In [12]:
model = ContentBasedRecommender(popularity_quantile=0.75)
model.fit(movies=movies, ratings=train)

results = evaluate_recommender(model=model, test_df=test, k=10, rating_threshold=0.0)

print("--- Test Evaluation Results ---")
for metric_name, score in results.items():
    print(f"{metric_name}: {score:.4f}")

--- Test Evaluation Results ---
Precision@10: 0.0005
Recall@10: 0.0049
NDCG@10: 0.0027
